# Evo 2 with Real Genomic DNA

Tests geometric stability of **Evo 2** (StripedHyena 2) on **real human genomic DNA** (chr22).

## Purpose

- **Synthetic DNA (Evo2_Scaling_Experiment)**: Tests pure architectural properties
- **Real DNA (this notebook)**: Ecological validity on actual human genome

Evo 2 was trained on a multi-species genomic corpus, so real DNA tests whether
its multi-hybrid architecture (interleaved Hyena conv + attention) maintains
geometric stability on realistic sequence composition, GC content variation,
repetitive elements, and functional structure.

## Checkpoints (7B family)

| Checkpoint     | Params | Context | Hardware        |
|----------------|--------|---------|-----------------|
| `evo2_7b_base` | 7B     | 8K      | A100 40GB (bf16) |
| `evo2_7b_262k` | 7B     | 262K    | A100 40GB (bf16) |
| `evo2_7b`      | 7B     | 1M      | A100 40GB (bf16) |

## Setup

1. Upload `evaluation_harness.py` and `perturbation_protocol.py` to `/content/`
2. Change Runtime to **GPU** (A100 recommended)
3. Run cells in order

---

In [ ]:
# Install Dependencies
#
# All 7B models run locally on A100 via the evo2 (Vortex) package.
# No API key or Transformer Engine required.

print('Installing core dependencies...')
!pip install -q shesha-geometry matplotlib seaborn pandas scipy
!pip install -q ninja
import sys, os

# Pin torch to 2.7.1+cu128 (Arc Institute recommended for evo2/flash-attn)
print('Pinning torch to 2.7.1+cu128...')
!pip install -q torch==2.7.1 --index-url https://download.pytorch.org/whl/cu128

import torch
print(f'torch {torch.__version__} | CUDA {torch.version.cuda}')

# Build flash-attn from source (~10-15 min on A100, one-time cost)
print('Building flash-attn 2.8.0.post2 from source (~10-15 min)...')
!pip install flash-attn==2.8.0.post2 --no-build-isolation

!pip install -q evo2

print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

from evo2 import Evo2
print('\nEvo2 package loaded successfully')
print('Done!')


In [ ]:
# Configuration

PHASE = 'full'  # 'quick' or 'full'

import os, sys
sys.path.insert(0, '.')

OUTPUT_BASE = './results/evo2_realdna_experiment/'
RESULTS_DIR = OUTPUT_BASE + 'results'
CACHE_DIR   = OUTPUT_BASE + 'cache'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

# 7B checkpoints only -- all run locally on A100 (bf16, no FP8 needed)
# (name, param_B, context_tokens, label)
ALL_CHECKPOINTS = [
    ('evo2_7b_base', 7.0,  8192,    'Evo2-7B-8K'),
    ('evo2_7b_262k', 7.0,  262144,  'Evo2-7B-262K'),
    ('evo2_7b',      7.0,  1048576, 'Evo2-7B-1M'),
]

# Embedding layer at ~75% model depth (per Evo 2 paper)
EMBEDDING_LAYERS = {
    'evo2_7b_base': 'blocks.28.mlp.l3',
    'evo2_7b_262k': 'blocks.28.mlp.l3',
    'evo2_7b':      'blocks.28.mlp.l3',
}

BATCH_OVERRIDES = {
    'evo2_7b_base': 4,
    'evo2_7b_262k': 4,
    'evo2_7b':      2,
}

CONFIG = {
    'quick': {
        'n_sequences': 500,
        'seq_length': 1000,
        'batch_size': 4,
        'n_bootstrap': 0,
        'snp_rates': [0.01, 0.05, 0.10],
    },
    'full': {
        'n_sequences': 10000,
        'seq_length': 1000,
        'batch_size': 4,
        'n_bootstrap': 5,
        'snp_rates': [0.01, 0.02, 0.05, 0.10],
    },
}

active_checkpoints = ALL_CHECKPOINTS if torch.cuda.is_available() else []
if not torch.cuda.is_available():
    print('WARNING: no GPU detected — no checkpoints will run')

config = CONFIG[PHASE]
print(f'Phase: {PHASE.upper()}')
print(f'Architecture: Evo 2 (StripedHyena 2) — 7B family')
print(f'Data: Real human genomic DNA (chr22)')
print(f'Sequences: {config["n_sequences"]}')
print(f'Active checkpoints: {len(active_checkpoints)}')
for ckpt_name, pb, cl, lbl in active_checkpoints:
    print(f'  {lbl}  ({pb:.0f}B, {cl:,} ctx)')
print(f'Results: {RESULTS_DIR}')


In [ ]:
# Download & Sample Real Genomic DNA from chr22

import urllib.request
import gzip
import numpy as np

CHR22_URL = 'https://hgdownload.soe.ucsc.edu/goldenPath/hg38/chromosomes/chr22.fa.gz'
VALID_BASES = set('ACGT')


def download_and_sample_genomic_dna(n_sequences, seq_length, seed=320):
    """
    Download human chr22 and sample non-overlapping regions.
    Filters out regions with >10% N bases.
    """
    cache_file = f'{CACHE_DIR}/chr22_sample_{n_sequences}_{seq_length}.txt'

    if os.path.exists(cache_file):
        print(f'Loading cached genomic sequences: {cache_file}')
        with open(cache_file, 'r') as f:
            sequences = [line.strip() for line in f if line.strip()]
        print(f'Loaded {len(sequences)} cached sequences')
        return sequences

    print('Downloading human chr22 (~50MB, first run only)...')
    os.makedirs(CACHE_DIR, exist_ok=True)

    with urllib.request.urlopen(CHR22_URL) as response:
        with gzip.GzipFile(fileobj=response) as f:
            lines = f.read().decode('utf-8').split('\n')
            chr22_seq = ''.join(line.strip() for line in lines[1:] if line.strip())

    print(f'Downloaded chr22 ({len(chr22_seq):,} bp)')

    rng = np.random.default_rng(seed)
    max_attempts = int(n_sequences * 1.5)
    sequences = []

    for _ in range(max_attempts):
        if len(sequences) >= n_sequences:
            break
        start = rng.integers(0, len(chr22_seq) - seq_length)
        seq = chr22_seq[start:start + seq_length].upper()

        n_count = sum(1 for c in seq if c not in VALID_BASES)
        if n_count < seq_length * 0.10:
            seq_list = list(seq)
            for i, c in enumerate(seq_list):
                if c not in VALID_BASES:
                    seq_list[i] = rng.choice(list('ACGT'))
            sequences.append(''.join(seq_list))

    if len(sequences) < n_sequences:
        print(f'WARNING: only found {len(sequences)}/{n_sequences} valid regions')

    with open(cache_file, 'w') as f:
        for seq in sequences:
            f.write(seq + '\n')

    print(f'Sampled {len(sequences)} real genomic sequences ({seq_length:,} bp each)')
    return sequences


# Sample real genomic DNA
sequences = download_and_sample_genomic_dna(
    config['n_sequences'], config['seq_length'], seed=320,
)

print(f'\nSequence stats:')
print(f'  Count: {len(sequences)}')
print(f'  Length: {len(sequences[0])}')
print(f'  Sample: {sequences[0][:60]}...')

# Base composition
total_bases = sum(len(s) for s in sequences)
print(f'\nBase composition (real chr22):')
for base in 'ACGT':
    count = sum(s.count(base) for s in sequences)
    print(f'  {base}: {count/total_bases*100:.1f}%')

In [ ]:
# DNA Perturbation Suite

from dataclasses import dataclass, field

DNA_BASES = ['A', 'C', 'G', 'T']
COMPLEMENT = {'A': 'T', 'T': 'A', 'C': 'G', 'G': 'C'}


def mutate_dna(sequence, mutation_rate, rng):
    seq = list(sequence)
    n_mutations = max(1, int(len(seq) * mutation_rate))
    positions = rng.choice(len(seq), size=n_mutations, replace=False)
    for pos in positions:
        original = seq[pos]
        alternatives = [b for b in DNA_BASES if b != original]
        seq[pos] = rng.choice(alternatives)
    return ''.join(seq)


def reverse_complement(sequence):
    return ''.join(COMPLEMENT.get(b, b) for b in reversed(sequence))


@dataclass
class PerturbedSet:
    name: str
    category: str
    sequences: list
    params: dict = field(default_factory=dict)
    description: str = ''


class DNAPerturbationSuite:
    def __init__(self, seed=320, snp_rates=None, include_rc=True):
        self.rng = np.random.default_rng(seed)
        self.snp_rates = snp_rates or [0.01, 0.02, 0.05, 0.10]
        self.include_rc = include_rc

    def run_all(self, sequences):
        results = {}
        for rate in self.snp_rates:
            name = f'snp_{int(rate * 100)}pct'
            perturbed = [mutate_dna(s, rate, self.rng) for s in sequences]
            results[name] = PerturbedSet(
                name=name, category='snp', sequences=perturbed,
                params={'mutation_rate': rate},
                description=f'SNP mutations at {rate*100:.0f}% rate',
            )
        if self.include_rc:
            results['reverse_complement'] = PerturbedSet(
                name='reverse_complement', category='rc',
                sequences=[reverse_complement(s) for s in sequences],
                params={}, description='Reverse complement transformation',
            )
        return results


suite = DNAPerturbationSuite(seed=320, snp_rates=config['snp_rates'])

test_perturbed = suite.run_all(sequences[:5])
for name, pset in test_perturbed.items():
    dists = [
        sum(a != b for a, b in zip(o, p)) / len(o)
        for o, p in zip(sequences[:5], pset.sequences)
    ]
    print(f'  {name}: mean_hamming={np.mean(dists):.4f}')
print('Perturbation suite ready')

In [ ]:
# Evo 2 Local Embedding Wrapper (7B via Vortex)

import torch
import gc
import numpy as np
from evo2 import Evo2


def cleanup_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        mem = torch.cuda.memory_allocated() / 1e9
        print(f'  GPU memory after cleanup: {mem:.2f} GB')


def make_evo2_embedding_fn(checkpoint_name, batch_size=4):
    """Load an Evo 2 checkpoint via Vortex and return an embedding function."""
    print(f'Loading {checkpoint_name}...')
    device = 'cuda:0'
    effective_batch = BATCH_OVERRIDES.get(checkpoint_name, batch_size)

    evo2_model = Evo2(checkpoint_name)
    n_params = sum(p.numel() for p in evo2_model.model.parameters()) / 1e6
    mem = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0
    print(f'  Params: {n_params:.1f}M | GPU mem: {mem:.2f} GB')

    layer_name = EMBEDDING_LAYERS[checkpoint_name]
    print(f'  Embedding layer: {layer_name}')

    @torch.no_grad()
    def embed(sequences):
        embeddings = []
        n_total = len(sequences)
        for idx, seq in enumerate(sequences):
            if (idx + 1) % 50 == 0 or idx == 0 or idx == n_total - 1:
                print(f'    Seq {idx+1}/{n_total}', end='\r')
            input_ids = torch.tensor(
                evo2_model.tokenizer.tokenize(seq), dtype=torch.int,
            ).unsqueeze(0).to(device)
            _, layer_embs = evo2_model(
                input_ids, return_embeddings=True, layer_names=[layer_name],
            )
            emb = layer_embs[layer_name]
            embeddings.append(emb.mean(dim=1).squeeze(0).cpu().float().numpy())
        print()
        return np.array(embeddings, dtype=np.float32)

    return embed, evo2_model, n_params


print('Evo 2 local (Vortex) wrapper ready — 7B checkpoints on A100')


In [ ]:
# Evaluation Harness (shesha-geometry)

from evaluation_harness import StabilityHarness

harness = StabilityHarness(
    window_size=0,
    metric='cosine',
    n_splits=30,
    seed=320,
    max_samples=2500,
    n_bootstrap=config['n_bootstrap'],
)

print(f'Shesha StabilityHarness configured (bootstrap={config["n_bootstrap"]})')

In [ ]:
# Run Experiment on Real Genomic DNA
#
# Same dual-pipeline as the scaling experiment,
# but using real chr22 sequences instead of synthetic.

import time
import pandas as pd
from evaluation_harness import ModelReport

print('=' * 70)
print(f'EVO 2 + REAL GENOMIC DNA (chr22) -- Phase: {PHASE.upper()}')
print('=' * 70)

reports = []
model_param_counts = []
all_detailed_rows = []

for ckpt_idx, (ckpt_name, params_b, ctx_len, label) in enumerate(active_checkpoints):
    print(f"\n{'='*70}")
    print(f'[{ckpt_idx+1}/{len(active_checkpoints)}] {label} ({ckpt_name})')
    print(f'  Params: {params_b:.0f}B | Context: {ctx_len:,}')
    print(f'  Data: Real human chr22 genomic DNA')
    print('=' * 70)

    start_time = time.time()

    embed_fn, evo2_model_obj, n_params_m = make_evo2_embedding_fn(
        ckpt_name, config['batch_size']
    )
    model_param_counts.append(n_params_m)

    print('\nGenerating perturbations on real DNA...')
    perturbed_sets = suite.run_all(sequences)

    safe_name = ckpt_name.replace('/', '_').replace('-', '_') + '_realdna'
    cache_clean = f'{CACHE_DIR}/{safe_name}_clean.npy'

    # Clean embeddings
    if os.path.exists(cache_clean):
        print(f'Loading cached clean embeddings: {cache_clean}')
        embeddings_clean = np.load(cache_clean)
    else:
        print(f'Computing clean embeddings ({len(sequences)} real DNA sequences)...')
        embeddings_clean = embed_fn(sequences)
        np.save(cache_clean, embeddings_clean)
        print(f'  Cached to {cache_clean}')
    print(f'  Shape: {embeddings_clean.shape}')

    # Perturbed embeddings
    perturbed_embeddings = {}
    for pert_name, pset in perturbed_sets.items():
        cache_pert = f'{CACHE_DIR}/{safe_name}_{pert_name}.npy'
        if os.path.exists(cache_pert):
            print(f'  Loading cached: {pert_name}')
            perturbed_embeddings[pert_name] = np.load(cache_pert)
        else:
            print(f'  Embedding: {pert_name}...')
            perturbed_embeddings[pert_name] = embed_fn(pset.sequences)
            np.save(cache_pert, perturbed_embeddings[pert_name])

    # Free GPU memory (only for local models)
    if evo2_model_obj is not None:
        del evo2_model_obj
        cleanup_gpu()

    # Run Shesha evaluation
    print('\nRunning Shesha evaluation...')
    report = harness.evaluate_all_perturbations(
        model_name=f'{label}-RealDNA',
        embeddings_clean=embeddings_clean,
        perturbed_dict=perturbed_embeddings,
    )
    reports.append(report)

    for pert_name, metrics in report.perturbation_breakdown().items():
        all_detailed_rows.append({
            'checkpoint': ckpt_name,
            'label': label,
            'params_B': params_b,
            'size_M': round(n_params_m, 1),
            'context_length': ctx_len,
            'data_type': 'real_chr22',
            'perturbation': pert_name,
            'rdm_similarity': metrics.get('rdm_similarity_score', 0),
            'rdm_drift': metrics.get('rdm_drift', 0),
            'pert_stability': metrics.get('perturbation_stability_score', 0),
            'pert_magnitude': metrics.get('perturbation_magnitude', 0),
            'composite': metrics.get('composite_stability', 0),
        })

    elapsed = time.time() - start_time
    summary = report.summary()
    print(f'\nCompleted {label} (Real DNA) in {elapsed/60:.1f} min')
    print(f'  Composite Stability: {summary["mean_composite_stability"]:.4f}')
    print(f'  RDM Similarity:      {summary["mean_rdm_similarity_score"]:.4f}')
    print(f'  Pert. Stability:     {summary["mean_perturbation_stability_score"]:.4f}')

# Save
df_detailed = pd.DataFrame(all_detailed_rows)
detailed_path = f'{RESULTS_DIR}/evo2_realdna_{PHASE}_detailed.csv'
df_detailed.to_csv(detailed_path, index=False)
print(f'\nSaved to {detailed_path}')
print(df_detailed.to_string(index=False))

print('\n' + '=' * 70)
print('EXPERIMENT COMPLETE')
print('=' * 70)

In [ ]:
# Visualization -- Parameter Scaling on Real DNA

import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams.update({'font.size': 12})

df_8k = df_detailed[df_detailed['context_length'] == 8192].copy()
df_1m = df_detailed[df_detailed['context_length'] == 1048576].copy()

fig, axes = plt.subplots(1, 3, figsize=(17, 5.5))

for col_idx, (metric_col, ylabel, color, title) in enumerate([
    ('composite', 'Composite Stability', '#2563EB', 'A. Composite Stability'),
    ('rdm_similarity', 'RDM Similarity', '#16A34A', 'B. RDM Similarity'),
    ('pert_magnitude', 'Perturbation Magnitude', '#DC2626', 'C. Perturbation Magnitude'),
]):
    ax = axes[col_idx]
    if len(df_8k) > 0:
        agg = df_8k.groupby('params_B').agg(
            mean=(metric_col, 'mean'), std=(metric_col, 'std'),
        ).reset_index().sort_values('params_B')
        ax.errorbar(agg['params_B'], agg['mean'], yerr=agg['std'],
                    fmt='o-', color=color, linewidth=2.5, markersize=10,
                    capsize=5, label='8K context (base)')
    if len(df_1m) > 0:
        agg2 = df_1m.groupby('params_B').agg(
            mean=(metric_col, 'mean'), std=(metric_col, 'std'),
        ).reset_index().sort_values('params_B')
        ax.errorbar(agg2['params_B'], agg2['mean'], yerr=agg2['std'],
                    fmt='s--', color=color, linewidth=2, markersize=9,
                    capsize=5, alpha=0.7, label='1M context')
    ax.set_xscale('log')
    ax.set_xlabel('Parameters (B)')
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontweight='bold')
    ax.grid(True, alpha=0.2)
    ax.legend(fontsize=9)

fig.suptitle('Evo 2 + Real DNA (chr22): Stability vs. Parameter Count',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/evo2_realdna_param_scaling_{PHASE}.png',
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# Context-Window Scaling on Real DNA (7B family)

df_7b = df_detailed[df_detailed['params_B'] == 7.0].copy()

if len(df_7b) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(17, 5.5))
    for col_idx, (metric_col, ylabel, color, title) in enumerate([
        ('composite', 'Composite Stability', '#7C3AED', 'A. Composite Stability'),
        ('rdm_similarity', 'RDM Similarity', '#0891B2', 'B. RDM Similarity'),
        ('pert_magnitude', 'Perturbation Magnitude', '#EA580C', 'C. Perturbation Magnitude'),
    ]):
        ax = axes[col_idx]
        agg = df_7b.groupby('context_length').agg(
            mean=(metric_col, 'mean'), std=(metric_col, 'std'),
        ).reset_index().sort_values('context_length')
        ax.errorbar(agg['context_length'], agg['mean'], yerr=agg['std'],
                    fmt='D-', color=color, linewidth=2.5, markersize=10, capsize=5)
        ax.set_xscale('log')
        ax.set_xlabel('Context Window (tokens)')
        ax.set_ylabel(ylabel)
        ax.set_title(title, fontweight='bold')
        ax.grid(True, alpha=0.2)
    fig.suptitle('Evo 2 (7B) + Real DNA: Stability vs. Context Window',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(f'{RESULTS_DIR}/evo2_realdna_context_scaling_{PHASE}.png',
                dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
else:
    print('No 7B checkpoints available -- skipping context-window plot')

In [ ]:
# Synthetic vs. Real DNA Comparison
#
# Compare stability on synthetic vs real DNA at matching checkpoints.

import glob

synth_base = './results/evo2_scaling_experiment/results'
synth_files = glob.glob(f'{synth_base}/evo2_scaling_*_detailed.csv')

if synth_files:
    df_synth = pd.read_csv(synth_files[0])

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Panel 1: Parameter scaling comparison (8K context)
    ax = axes[0]
    synth_8k = df_synth[df_synth['context_length'] == 8192]
    real_8k = df_detailed[df_detailed['context_length'] == 8192]

    if len(synth_8k) > 0:
        agg_s = synth_8k.groupby('params_B')['composite'].agg(['mean', 'std']).reset_index()
        ax.errorbar(agg_s['params_B'], agg_s['mean'], yerr=agg_s['std'],
                    fmt='s--', color='#9CA3AF', linewidth=2, markersize=9,
                    capsize=5, label='Synthetic DNA')
    if len(real_8k) > 0:
        agg_r = real_8k.groupby('params_B')['composite'].agg(['mean', 'std']).reset_index()
        ax.errorbar(agg_r['params_B'], agg_r['mean'], yerr=agg_r['std'],
                    fmt='o-', color='#2563EB', linewidth=2.5, markersize=10,
                    capsize=5, label='Real DNA (chr22)')

    ax.set_xscale('log')
    ax.set_xlabel('Parameters (B)', fontsize=12)
    ax.set_ylabel('Composite Stability', fontsize=12)
    ax.set_title('Evo 2 (8K): Synthetic vs Real DNA', fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.2)

    # Panel 2: Context-window comparison (7B)
    ax = axes[1]
    synth_7b = df_synth[df_synth['params_B'] == 7.0]
    real_7b = df_detailed[df_detailed['params_B'] == 7.0]

    if len(synth_7b) > 0:
        agg_s7 = synth_7b.groupby('context_length')['composite'].agg(['mean', 'std']).reset_index()
        ax.errorbar(agg_s7['context_length'], agg_s7['mean'], yerr=agg_s7['std'],
                    fmt='s--', color='#9CA3AF', linewidth=2, markersize=9,
                    capsize=5, label='Synthetic DNA')
    if len(real_7b) > 0:
        agg_r7 = real_7b.groupby('context_length')['composite'].agg(['mean', 'std']).reset_index()
        ax.errorbar(agg_r7['context_length'], agg_r7['mean'], yerr=agg_r7['std'],
                    fmt='D-', color='#7C3AED', linewidth=2.5, markersize=10,
                    capsize=5, label='Real DNA (chr22)')

    ax.set_xscale('log')
    ax.set_xlabel('Context Window (tokens)', fontsize=12)
    ax.set_ylabel('Composite Stability', fontsize=12)
    ax.set_title('Evo 2 (7B): Synthetic vs Real DNA', fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.2)

    plt.tight_layout()
    plt.savefig(f'{RESULTS_DIR}/evo2_real_vs_synth_{PHASE}.png',
                dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f'Saved to {RESULTS_DIR}/evo2_real_vs_synth_{PHASE}.png')
else:
    print('No synthetic experiment results found.')
    print('Run Evo2_Scaling_Experiment.ipynb first for comparison.')

In [ ]:
# Per-Perturbation Heatmap

fig, axes = plt.subplots(1, 2, figsize=(16, max(5, len(active_checkpoints))))

col_order = [lbl for _, _, _, _, lbl in active_checkpoints]

pivot_comp = df_detailed.pivot_table(
    values='composite', index='perturbation', columns='label',
)
pivot_comp = pivot_comp[[c for c in col_order if c in pivot_comp.columns]]

ax = axes[0]
sns.heatmap(pivot_comp, annot=True, fmt='.3f', cmap='RdYlGn', vmin=0, vmax=1,
            ax=ax, linewidths=0.5, cbar_kws={'label': 'Composite'})
ax.set_title('Composite Stability (Real DNA)', fontweight='bold')
ax.set_xlabel('Evo 2 Checkpoint')
ax.set_ylabel('Perturbation')

pivot_rdm = df_detailed.pivot_table(
    values='rdm_similarity', index='perturbation', columns='label',
)
pivot_rdm = pivot_rdm[[c for c in col_order if c in pivot_rdm.columns]]

ax = axes[1]
sns.heatmap(pivot_rdm, annot=True, fmt='.3f', cmap='RdYlGn', vmin=0, vmax=1,
            ax=ax, linewidths=0.5, cbar_kws={'label': 'RDM Similarity'})
ax.set_title('RDM Similarity (Real DNA)', fontweight='bold')
ax.set_xlabel('Evo 2 Checkpoint')
ax.set_ylabel('Perturbation')

fig.suptitle('Evo 2 + Real DNA (chr22): Per-Perturbation Breakdown',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/evo2_realdna_{PHASE}_heatmap.png',
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()